# Step 01B — Validator-aligned AP and bootstrap audit

**Purpose:** resolve the internal AP reconstruction mismatch from Step 01, using the same **Ultralytics validation path** for native mAP, exported true-positive matching flags, fixed-confidence counts, confidence curves and bootstrap CIs.

- **No training, fine-tuning, threshold optimization, or annotation editing.**
- Requires the same SHA-256 locked checkpoint and the same 30 internal + 66 external images.
- Captures per-image matching statistics with Ultralytics `on_val_batch_end` callback, replays them through the official `ap_per_class` method and **asserts exact AP parity** before bootstrapping.
- Uses a *separate validation pass* at confidence 0.25 for fixed-threshold TP/FP/FN; never treats confidence-integrated AP as a fixed-threshold recall.
- `0.001` is used only for AP and PR-curve input; `0.25` remains the prespecified operating point.
- This is a **working technical audit**, not itself proof of clinical annotation equivalence or recovery of the original manuscript checkpoint.

In [ ]:
%pip install -q ultralytics==8.4.139 numpy pandas matplotlib pyyaml pillow

## A. Mount Drive and lock inputs

The script aborts if the checkpoint has changed. Two evaluation passes use the same 66 external and 30 internal original files as Step 01.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, zipfile, sys, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch, ultralytics
import matplotlib.pyplot as plt
from ultralytics import YOLO
from ultralytics.utils.metrics import ap_per_class

assert ultralytics.__version__ == '8.4.139', ultralytics.__version__
ROOT=Path('/content/drive/MyDrive/Pediatric_Caries_AI')
WEIGHTS=ROOT/'final_model/YOLO11s_Final/final_60epochs.pt'
EXPECTED_HASH='19ce0e2ae9731a9acb5baeaa4cb514782aa182e7d916d8a101298dd59cdb077e'
DATA={
 'internal': ROOT/'processed_data/yolo_caries_baseline',
 'external': ROOT/'external_validation/MOPG7_66',
}
EXPECTED={'internal':(30,108), 'external':(66,55)}
IMGSZ=1024; NMS_IOU=0.70; MAX_DET=300; BOOTSTRAPS=2000; SEED=42
DEVICE=0 if torch.cuda.is_available() else 'cpu'
def sha256(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda: f.read(4*1024*1024),b''): h.update(b)
 return h.hexdigest()
assert WEIGHTS.exists(), WEIGHTS
assert sha256(WEIGHTS)==EXPECTED_HASH, 'STOP: Checkpoint identity does not match audited snapshot.'
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=ROOT/'reviewer_revision'/f'validator_parity_{STAMP}'
OUT.mkdir(parents=True,exist_ok=False)
print('Weights SHA256:',EXPECTED_HASH,'Device:',DEVICE,'Ultralytics:',ultralytics.__version__)
print('Output:',OUT)

# Re-create the exact data YAML structure used by Step 01; no image files are changed.
import yaml
manifest=[]
for cohort,folder in DATA.items():
 images=sorted((folder/'images/test').glob('*'))
 images=[p for p in images if p.suffix.lower() in ('.jpg','.jpeg','.png','.tif','.tiff','.bmp')]
 assert len(images)==EXPECTED[cohort][0], (cohort,len(images))
 count=0
 for p in images:
  lp=folder/'labels/test'/f'{p.stem}.txt'
  assert lp.exists(),lp
  labels=[line for line in lp.read_text().splitlines() if line.strip()]
  for line in labels:
   nums=line.split(); assert len(nums)==5 and int(float(nums[0]))==0,(lp,line)
  count+=len(labels)
  manifest.append(dict(cohort=cohort,image=p.name,n_gt=len(labels),image_sha256=sha256(p),label_sha256=sha256(lp)))
 assert count==EXPECTED[cohort][1],(cohort,count)
 spec={'path':str(folder),'train':'images/test','val':'images/test','test':'images/test','names':{0:'caries'},'nc':1}
 (OUT/f'{cohort}.yaml').write_text(yaml.safe_dump(spec,sort_keys=False))
print('Validated cohort counts and checkpoint hash.')
manifest=pd.DataFrame(manifest)
manifest.to_csv(OUT/'input_manifest_sha256.csv',index=False)

## B. Capture Ultralytics validator statistics, image by image

Unlike the Step 01 reconstructed AP, this uses the same validation batches and ground-truth matching flags as `model.val()`. The callback is read-only and does not modify detection outputs.

In [ ]:
def as_numpy(x):
 if isinstance(x,torch.Tensor): return x.detach().cpu().numpy().copy()
 return np.asarray(x).copy()

def run_validator(cohort,conf,name):
 captured=[]
 state={'previous':0}
 def capture_validator(v):
  # Ultralytics 8.4.139: DetMetrics.stats is a dict of per-image arrays.
  s=v.metrics.stats
  names=list(v.metrics.box.image_metrics.keys())
  n=len(s['tp'])
  assert all(len(s[k])==n for k in ['conf','pred_cls','target_cls','target_img']),    'Inconsistent per-image validator statistic counts'
  assert len(names)==n,('Image-name ordering does not match validator stats',len(names),n)
  for i in range(state['previous'],n):
   tp=as_numpy(s['tp'][i]).astype(bool).reshape(-1,10)
   scores=as_numpy(s['conf'][i]).astype(float).reshape(-1)
   pc=as_numpy(s['pred_cls'][i]).astype(int).reshape(-1)
   target=as_numpy(s['target_cls'][i]).astype(int).reshape(-1)
   assert len(tp)==len(scores)==len(pc)
   assert np.all(pc==0) and np.all(target==0)
   captured.append(dict(image=names[i],tp=tp,scores=scores,pred_cls=pc,n_gt=len(target)))
  state['previous']=n
 model=YOLO(str(WEIGHTS))
 assert model.names=={0:'caries'},model.names
 model.add_callback('on_val_batch_end',capture_validator)
 metrics=model.val(data=str(OUT/f'{cohort}.yaml'),split='test',imgsz=IMGSZ,conf=conf,
                   iou=NMS_IOU,max_det=MAX_DET,device=DEVICE,half=False,workers=0,batch=4,
                   save=False,save_txt=False,save_json=False,plots=False,
                   project=str(OUT/'validation_runs'),name=name,exist_ok=False,verbose=False)
 exp_sub=manifest[manifest.cohort==cohort]
 assert len(captured)==len(exp_sub),('Images not fully captured',cohort,len(captured),len(exp_sub))
 assert len(set(x['image'] for x in captured))==len(captured),'Duplicate validator image names'
 assert set(x['image'] for x in captured)==set(exp_sub.image)
 gt_by_image=dict(zip(exp_sub.image,exp_sub.n_gt))
 for x in captured:
  assert x['n_gt']==gt_by_image[x['image']],('Ground-truth mismatch',cohort,x['image'])
 return captured,{'cohort':cohort,'conf':conf,'mAP50':float(metrics.box.map50),
                  'mAP50_95':float(metrics.box.map),'precision':float(metrics.box.mp),
                  'recall':float(metrics.box.mr)}

stats={}; official=[]
for cohort in DATA:
 for conf,tag in [(0.001,'low'),(0.25,'fixed')]:
  print('Validating',cohort,'at conf',conf)
  items,metric=run_validator(cohort,conf,f'{cohort}_{tag}')
  stats[(cohort,tag)]=items
  official.append(metric)
  print('  AP50=',metric['mAP50'],'AP50:95=',metric['mAP50_95'],
        'predictions=',sum(len(x['scores']) for x in items))
official=pd.DataFrame(official)
official.to_csv(OUT/'native_validation_metrics_by_conf.csv',index=False)
print('All four official validation passes captured.')

## C. Recompute AP from official per-image matching statistics

This is a required consistency gate. If the recomputation differs from `model.val()`, do not cite reconstructed per-IoU values or their CIs as validated.

In [ ]:
IOU_LEVELS=np.round(np.arange(.50,.951,.05),2)
def replay_native_ap(items):
 tp=np.concatenate([x['tp'] for x in items],axis=0)
 conf=np.concatenate([x['scores'] for x in items])
 pc=np.concatenate([x['pred_cls'] for x in items])
 gt=np.zeros(sum(x['n_gt'] for x in items),dtype=int)
 if len(gt)==0: return np.full(10,np.nan)
 if len(conf)==0: return np.zeros(10)
 ap=ap_per_class(tp,conf,pc,gt,plot=False)[5]
 assert ap.shape==(1,10),ap.shape
 return ap[0]

parity=[]; ap_rows=[]
for cohort in DATA:
 vals=replay_native_ap(stats[(cohort,'low')])
 target=official[(official.cohort==cohort)&(official.conf==.001)].iloc[0]
 delta50=abs(float(vals[0])-float(target.mAP50))
 delta95=abs(float(np.mean(vals))-float(target.mAP50_95))
 parity.append(dict(cohort=cohort,native_mAP50=float(target.mAP50),replayed_mAP50=float(vals[0]),
                    delta50=delta50,native_mAP50_95=float(target.mAP50_95),
                    replayed_mAP50_95=float(np.mean(vals)),delta50_95=delta95,
                    parity_pass=bool(delta50<=1e-5 and delta95<=1e-5)))
 for i,(thr,v) in enumerate(zip(IOU_LEVELS,vals)):
  ap_rows.append(dict(cohort=cohort,iou=float(thr),AP_validator_aligned=float(v)))
parity_df=pd.DataFrame(parity)
parity_df.to_csv(OUT/'mandatory_ap_parity_check.csv',index=False)
pd.DataFrame(ap_rows).to_csv(OUT/'ap_by_iou_validator_aligned.csv',index=False)
display(parity_df)
PARITY_OK=bool(parity_df.parity_pass.all())
if not PARITY_OK:
 print('STOP: official AP reconstruction is not verified. Share this archive for diagnosis; do NOT adopt its AP CIs.')
else:
 print('PASS: AP reconstruction agrees with native Ultralytics official output at both endpoints.')

## D. Confirm fixed-threshold confusion counts from the validator (not predictor)

A second `model.val(conf=0.25)` pass provides actual matching at the prespecified operating point, rather than filtering previously matched low-confidence predictions. The all-positive baseline is computed on the same 66 cases.

In [ ]:
def safe_rate(a,b): return float(a/b) if b else np.nan
fixed_per=[];fixed_summaries=[]
for cohort in DATA:
 items=stats[(cohort,'fixed')]
 for x in items:
  nt=x['n_gt']; npr=len(x['scores']); tp=int(x['tp'][:,0].sum())
  fixed_per.append(dict(cohort=cohort,image=x['image'],n_gt=nt,n_predictions=npr,
                        TP=tp,FP=npr-tp,FN=nt-tp,
                        actual_positive=int(nt>0),pred_positive=int(npr>0)))
 d=pd.DataFrame([row for row in fixed_per if row['cohort']==cohort])
 tp=int(d.TP.sum());fp=int(d.FP.sum());fn=int(d.FN.sum())
 itp=int(((d.actual_positive==1)&(d.pred_positive==1)).sum())
 ifp=int(((d.actual_positive==0)&(d.pred_positive==1)).sum())
 itn=int(((d.actual_positive==0)&(d.pred_positive==0)).sum())
 ifn=int(((d.actual_positive==1)&(d.pred_positive==0)).sum())
 fixed_summaries.append(dict(cohort=cohort,lesion_TP=tp,lesion_FP=fp,lesion_FN=fn,
  lesion_precision=safe_rate(tp,tp+fp),lesion_recall=safe_rate(tp,tp+fn),
  lesion_F1=safe_rate(2*tp,2*tp+fp+fn),image_TP=itp,image_FP=ifp,image_TN=itn,image_FN=ifn,
  image_sensitivity=safe_rate(itp,itp+ifn),image_specificity=safe_rate(itn,itn+ifp),
  image_precision=safe_rate(itp,itp+ifp),image_accuracy=safe_rate(itp+itn,len(d)),
  image_F1=safe_rate(2*itp,2*itp+ifp+ifn)))
fixed_images=pd.DataFrame(fixed_per)
fixed_images.to_csv(OUT/'fixed_conf025_validator_per_image.csv',index=False)
fixed_summary=pd.DataFrame(fixed_summaries)
fixed_summary.to_csv(OUT/'fixed_conf025_validator_summary.csv',index=False)
display(fixed_summary)

ex=fixed_summary[fixed_summary.cohort=='external'].iloc[0]
base=pd.DataFrame([
 dict(method='Frozen YOLO11s at conf 0.25',TP=int(ex.image_TP),FP=int(ex.image_FP),TN=int(ex.image_TN),FN=int(ex.image_FN),
      sensitivity=ex.image_sensitivity,specificity=ex.image_specificity,
      precision=ex.image_precision,accuracy=ex.image_accuracy,F1=ex.image_F1),
 dict(method='All-positive classification baseline',TP=39,FP=27,TN=0,FN=0,
      sensitivity=1.,specificity=0.,precision=39/66,accuracy=39/66,F1=78/(78+27))])
base.to_csv(OUT/'external_image_level_baseline_validator_aligned.csv',index=False)
display(base)

## E. Bootstrap CIs only after AP parity

Resample radiographs and replay official per-image TP/score/target arrays with Ultralytics's own AP implementation. This represents sampling uncertainty conditional on the frozen model and current reference annotations, not uncertainty from retraining.

In [ ]:
ci_rows=[];boot_rows=[]
if PARITY_OK:
 for cohort in DATA:
  items=stats[(cohort,'low')];df=fixed_images[fixed_images.cohort==cohort].set_index('image')
  rng=np.random.default_rng(SEED)
  for rep in range(BOOTSTRAPS):
   ii=rng.integers(0,len(items),size=len(items))
   selected=[items[i] for i in ii]
   AP=replay_native_ap(selected)
   d=df.loc[[x['image'] for x in selected]]
   tp=int(d.TP.sum());fp=int(d.FP.sum());fn=int(d.FN.sum())
   itp=int(((d.actual_positive==1)&(d.pred_positive==1)).sum())
   ifp=int(((d.actual_positive==0)&(d.pred_positive==1)).sum())
   itn=int(((d.actual_positive==0)&(d.pred_positive==0)).sum())
   ifn=int(((d.actual_positive==1)&(d.pred_positive==0)).sum())
   boot_rows.append(dict(cohort=cohort,replicate=rep+1,AP50=float(AP[0]),AP50_95=float(np.nanmean(AP)),
    lesion_precision=safe_rate(tp,tp+fp),lesion_recall=safe_rate(tp,tp+fn),
    lesion_F1=safe_rate(2*tp,2*tp+fp+fn),
    image_sensitivity=safe_rate(itp,itp+ifn),image_specificity=safe_rate(itn,itn+ifp),
    image_accuracy=safe_rate(itp+itn,len(d)),image_precision=safe_rate(itp,itp+ifp),
    image_F1=safe_rate(2*itp,2*itp+ifp+ifn)))
  print('Bootstrap complete:',cohort)
 boot=pd.DataFrame(boot_rows)
 boot.to_csv(OUT/'validator_aligned_bootstrap_replicates.csv',index=False)
 for (cohort),s in boot.groupby('cohort'):
  for k in [c for c in s.columns if c not in ('cohort','replicate')]:
   x=s[k].dropna().to_numpy(float)
   lo,hi=np.quantile(x,[.025,.975]) if len(x) else (np.nan,np.nan)
   ci_rows.append(dict(cohort=cohort,metric=k,CI_low=lo,CI_high=hi,n_valid=len(x),n_total=BOOTSTRAPS))
 pd.DataFrame(ci_rows).to_csv(OUT/'validator_aligned_bootstrap_CI.csv',index=False)
 display(pd.DataFrame(ci_rows))
else:
 print('Bootstrap deliberately skipped because mandatory official AP parity failed.')

## F. PR curve, exact per-IoU AP and portable evidence archive

Only statistics from the validator-aligned low-confidence evaluation are plotted. Native AP parity is recorded in the archive. Old manuscript values remain separate.

In [ ]:
def draw_pr(cohort,items):
 tp=np.concatenate([x['tp'] for x in items],axis=0)[:,0]
 scores=np.concatenate([x['scores'] for x in items])
 ngt=sum(x['n_gt'] for x in items)
 ind=np.argsort(-scores,kind='stable')
 tpc=np.cumsum(tp[ind]);fpc=np.cumsum(~tp[ind]);r=tpc/ngt;p=tpc/np.maximum(tpc+fpc,1)
 return r,p
if PARITY_OK:
 plt.figure(figsize=(7,5))
 for cohort in DATA:
  r,p=draw_pr(cohort,stats[(cohort,'low')]); plt.plot(r,p,label=cohort.title())
 plt.xlabel('Recall');plt.ylabel('Precision');plt.title('IoU 0.50 PR curves — validator-matched data')
 plt.xlim(0,1);plt.ylim(0,1.02);plt.legend();plt.grid(alpha=.25)
 plt.tight_layout();plt.savefig(OUT/'precision_recall_validator_aligned.png',dpi=200);plt.show();plt.close()
 plt.figure(figsize=(7,4))
 for cohort in DATA:
  sub=pd.DataFrame(ap_rows).query('cohort == @cohort')
  plt.plot(sub.iou,sub.AP_validator_aligned,marker='o',label=cohort.title())
 plt.xlabel('Matching IoU');plt.ylabel('Official-validator AP');plt.title('AP by IoU — same official evaluation statistics')
 plt.legend();plt.grid(alpha=.25);plt.tight_layout()
 plt.savefig(OUT/'ap_by_iou_validator_aligned.png',dpi=200);plt.show();plt.close()

# Portable per-image statistics at both confidence settings, preserving negative images.
for tag in ('low','fixed'):
 rows=[]
 for cohort in DATA:
  for item in stats[(cohort,tag)]:
   if len(item['scores'])==0:
    rows.append(dict(cohort=cohort,image=item['image'],n_gt=item['n_gt'],pred_index=-1,conf=np.nan,**{f'tp_iou_{j}':np.nan for j in range(10)}))
   for i,s in enumerate(item['scores']):
    rows.append(dict(cohort=cohort,image=item['image'],n_gt=item['n_gt'],pred_index=i,conf=float(s),
     **{f'tp_iou_{j}':int(item['tp'][i,j]) for j in range(10)}))
 pd.DataFrame(rows).to_csv(OUT/f'validator_statistics_conf_{tag}.csv',index=False)

manifest_json={'checkpoint_sha256':EXPECTED_HASH,'ultralytics':ultralytics.__version__,
               'torch':torch.__version__,'python':sys.version,'device':str(DEVICE),
               'parameters':{'imgsz':IMGSZ,'NMS_iou':NMS_IOU,'max_det':MAX_DET,'batch':4,
                             'low_conf':.001,'fixed_conf':.25,'bootstrap_replicates':BOOTSTRAPS,'seed':SEED},
               'AP_parity_pass':PARITY_OK,
               'history_not_recovered':True,
               'annotation_definition_equivalence_not_confirmed':True}
(OUT/'run_provenance.json').write_text(json.dumps(manifest_json,indent=2))
(OUT/'README.txt').write_text('VALIDATOR-ALIGNED STEP 01B REPRODUCIBILITY AUDIT\n'
 'Only refer to AP bootstrap when mandatory_ap_parity_check.csv passes for both cohorts.\n'
 'Fixed threshold counts use separate model.val(conf=0.25) passes.\n'
 'Historical manuscript snapshot identity and external annotation clinical comparability remain unverified.\n')
zip_path=OUT/'Step_01B_Validator_Aligned_Results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in sorted(OUT.iterdir()):
  if p.is_file() and p!=zip_path: z.write(p,p.name)
print('AP parity passed:',PARITY_OK)
print('Download this file from Google Drive and upload it to the chat:')
print(zip_path)